# 06. Building Your First Complete Agent

This module acts as the capstone for the beginner track. We will build the exact same agent three times to understand the evolution of frameworks:
1. **Raw Agent Loop:** Manually handling OpenAI's API.
2. **LangGraph:** Using a state-machine architecture.
3. **PydanticAI:** Using a type-safe, lightweight Pythonic framework.

## Setup

Install the necessary libraries:

In [ ]:
!pip install openai langgraph langchain-openai pydantic-ai

### API Keys

We will use `getpass` to securely prompt for your OpenAI API Key.

In [ ]:
import os
import getpass

if 'OPENAI_API_KEY' not in os.environ:
    print('Enter your OpenAI API Key:')
    os.environ['OPENAI_API_KEY'] = getpass.getpass()

print('API Key loaded!')

## The Scenario & Tools

We are building a Support Escalation Agent. It has access to two tools:
1. `get_ticket_details(ticket_id)`: Fetches a complaint.
2. `get_billing_status(customer_id)`: Checks if their card is valid.

In [ ]:
# Our Mock Database Tools
def get_ticket_details(ticket_id: str) -> str:
    """Returns the details of a support ticket."""
    print(f"[TOOL EXECUTION] Fetching details for ticket {ticket_id}...")
    return f"Ticket {ticket_id}: Customer is angry that they were charged twice for their monthly subscription."

def get_billing_status(customer_id: str) -> str:
    """Returns the billing status of a customer."""
    print(f"[TOOL EXECUTION] Fetching billing status for customer {customer_id}...")
    return f"Customer {customer_id}: Account in good standing. Last charge was duplicated due to system error."

## Approach 1: The Raw Agent Loop (OpenAI Native)

Under the hood, an agent is just a `while` loop that calls the LLM, executes requested tools, and feeds the results back.

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_details",
            "description": "Returns the details of a support ticket.",
            "parameters": {
                "type": "object",
                "properties": {"ticket_id": {"type": "string"}},
                "required": ["ticket_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_billing_status",
            "description": "Returns the billing status of a customer.",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "string"}},
                "required": ["customer_id"]
            }
        }
    }
]

def raw_agent(prompt: str):
    messages = [{"role": "user", "content": prompt}]
    print("\n--- Starting Raw Agent Loop ---")
    
    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools
        )
        message = response.choices[0].message
        messages.append(message)
        
        if not message.tool_calls:
            print(f"\n[AGENT FINISHED] Final Answer: {message.content}")
            break
            
        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            print(f"\n[AGENT DECISION] Model wants to call {tool_call.function.name} with {args}")
            
            if tool_call.function.name == "get_ticket_details":
                result = get_ticket_details(args['ticket_id'])
            elif tool_call.function.name == "get_billing_status":
                result = get_billing_status(args['customer_id'])
                
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

try:
    raw_agent("What is the status of ticket T-102 and what is customer C-55's billing status? Summarize the situation.")
except Exception as e:
    print(f"API Error: {e}")

## Approach 2: LangGraph

Instead of a raw loop, LangGraph represents the agent as a state machine. It abstracts the `messages` array into a Graph State, and handles the tool execution automatically.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

# 1. Define Tools (using decorators this time)
@tool
def lg_get_ticket_details(ticket_id: str) -> str:
    """Returns the details of a support ticket."""
    print(f"[LANGGRAPH TOOL] Fetching ticket {ticket_id}")
    return get_ticket_details(ticket_id)

@tool
def lg_get_billing_status(customer_id: str) -> str:
    """Returns the billing status of a customer."""
    print(f"[LANGGRAPH TOOL] Fetching billing {customer_id}")
    return get_billing_status(customer_id)

lg_tools = [lg_get_ticket_details, lg_get_billing_status]
tool_node = ToolNode(lg_tools)

# 2. Define State
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 3. Define the LLM Node
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(lg_tools)

def call_model(state: State):
    print("\n[LANGGRAPH NODE] Calling Model...")
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def should_continue(state: State):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        print("[LANGGRAPH EDGE] Routing to Tools...")
        return "tools"
    print("[LANGGRAPH EDGE] Routing to END...")
    return END

# 4. Build the Graph
builder = StateGraph(State)
builder.add_node("agent", call_model)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])
builder.add_edge("tools", "agent")
graph = builder.compile()

try:
    print("\n--- Starting LangGraph Agent ---")
    response = graph.invoke({"messages": [("user", "What is the status of ticket T-102 and what is customer C-55's billing status?")]})
    print(f"\n[LANGGRAPH FINISHED] Final Answer: {response['messages'][-1].content}")
except Exception as e:
    print(f"API Error: {e}")

## Approach 3: PydanticAI

PydanticAI completely hides the graph and the raw loop, offering a clean, type-safe, decorator-based interface. It feels like writing standard Python.

In [ ]:
from pydantic_ai import Agent

# 1. Define Agent
pydantic_agent = Agent(
    'openai:gpt-4o',
    system_prompt='You are a helpful support escalation agent.',
)

# 2. Register Tools directly to the agent
@pydantic_agent.tool_plain
def pa_get_ticket_details(ticket_id: str) -> str:
    """Returns the details of a support ticket."""
    print(f"[PYDANTIC-AI TOOL] Fetching ticket {ticket_id}")
    return get_ticket_details(ticket_id)

@pydantic_agent.tool_plain
def pa_get_billing_status(customer_id: str) -> str:
    """Returns the billing status of a customer."""
    print(f"[PYDANTIC-AI TOOL] Fetching billing {customer_id}")
    return get_billing_status(customer_id)

try:
    print("\n--- Starting PydanticAI Agent ---")
    result = pydantic_agent.run_sync("What is the status of ticket T-102 and what is customer C-55's billing status?")
    print(f"\n[PYDANTIC-AI FINISHED] Final Answer: {result.data}")
except Exception as e:
    print(f"API Error: {e}")

## Conclusion

Notice how **all three frameworks accomplished the exact same task** with the exact same LLM capability. Frameworks do not make the LLM smarter. They just provide different Developer Experiences (DX) for managing state, routing, and tool execution.